In [1]:
%pip install pandas numpy scikit-learn xgboost shap matplotlib seaborn

Defaulting to user installation because normal site-packages is not writeable
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.8/556.8 kB 26.8 MB/s eta 0:00:00
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.8/28.8 MB 62.8 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
"""
guns_police_involvement_pipeline.py

Goal: Predict Police Involvement (Yes/No) from the Kaggle dataset:
https://www.kaggle.com/datasets/datatattle/guns-incident-data

Outputs:
 - Model training for Logistic Regression, RandomForest, XGBoost
 - Metrics: Accuracy, F1, ROC-AUC
 - SHAP summary plot for best model (if SHAP + XGBoost available)
 - Error analysis: subgroup false negative rates and underpredicted subgroups

Usage:
 - Place the dataset CSV in the same folder and set CSV_PATH if needed.
 - Run in a notebook or terminal.
"""

import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ---------- User edit point ----------
CSV_PATH = "guns.csv"   # set to your CSV file path (default tries 'guns.csv')
TARGET_COL = None       # If None -> auto-detect plausible target (see auto-detection)
# -------------------------------------

# Helper: try to auto-detect a police involvement column
def detect_target_column(df):
    possible = [c for c in df.columns if 'police' in c.lower() or 'officer' in c.lower() or 'police_involved' in c.lower() or 'policeinvolved' in c.lower() or 'policein' in c.lower() or 'police_involvement' in c.lower()]
    # also consider 'reported' or 'response' combos
    possible += [c for c in df.columns if ('involved' in c.lower() and 'police' in c.lower())]
    # unique common names
    for name in ['police_involved','police','police_reported','policeinvolved','police_interaction','police_response']:
        if name in df.columns:
            return name
    if possible:
        return possible[0]
    # fallback: try to find a binary-like column (0/1 or Yes/No) with police in values
    for c in df.columns:
        vals = df[c].dropna().unique()[:20].astype(str)
        if any(x.lower() in ('yes','no','y','n','true','false','1','0') for x in vals):
            # check if the column name hints at police
            if any(k in c.lower() for k in ['police','officer','response','reported']):
                return c
    return None

# Load CSV
if not os.path.exists(CSV_PATH):
    print(f"ERROR: CSV not found at '{CSV_PATH}'. Please place dataset CSV there or update CSV_PATH.")
    sys.exit(1)

df = pd.read_csv(CSV_PATH, low_memory=False)
print(f"Loaded dataset with shape: {df.shape}")

# show columns summary
print("\nColumns (first 40):")
print(df.columns[:40])

# Detect target
if TARGET_COL is None:
    TARGET_COL = detect_target_column(df)
    if TARGET_COL:
        print(f"\nAuto-detected target column: '{TARGET_COL}'")
    else:
        print("\nCould not auto-detect a 'police involvement' column. Please set TARGET_COL variable in the script to the correct column name and re-run.")
        # Show columns to help user decide
        print("Columns found:")
        for c in df.columns:
            print(" -", c)
        sys.exit(1)
else:
    print(f"\nUsing user-specified TARGET_COL: {TARGET_COL}")

# Inspect target values
print("\nTarget column value counts:")
print(df[TARGET_COL].value_counts(dropna=False).head(20))

# Convert target to binary (0/1)
def binarize_target(series):
    s = series.copy()
    s = s.fillna("unknown").astype(str).str.strip().str.lower()
    # common mappings
    true_vals = set(['yes','y','true','t','1','1.0','police','involved','involved_yes','responded','respond'])
    false_vals = set(['no','n','false','f','0','0.0','not involved','not','none','no response'])
    mapped = []
    for v in s:
        if v in true_vals or any(k in v for k in ['yes','police','respond','involved','responded']):
            mapped.append(1)
        elif v in false_vals or any(k in v for k in ['no','none','not']):
            mapped.append(0)
        else:
            # numeric?
            try:
                nv = float(v)
                if nv == 1.0:
                    mapped.append(1)
                elif nv == 0.0:
                    mapped.append(0)
                else:
                    mapped.append(np.nan)
            except:
                mapped.append(np.nan)
    return pd.Series(mapped, index=series.index).astype('float')

y = binarize_target(df[TARGET_COL])
print("\nBinarized target value counts (after mapping):")
print(y.value_counts(dropna=False))

# Drop rows with unknown target
mask_valid = y.isin([0.0,1.0])
print(f"\nKeeping {mask_valid.sum()} rows with valid target out of {len(y)}")
df = df.loc[mask_valid].copy()
y = y.loc[mask_valid].astype(int)

# Basic feature selection:
# - We'll take columns that are non-id, non-text heavy. But include common contextual/demographic columns if present.
exclude_prefixes = ['id', 'case', 'report', 'unique', 'url', 'photo']
text_like = []
for c in df.columns:
    if df[c].dtype == object and df[c].str.len().fillna(0).mean() > 200:
        text_like.append(c)
# Candidate features: keep columns that are not the target and not overwhelmingly text
feature_candidates = [c for c in df.columns if c != TARGET_COL and c not in text_like]
# Remove columns that are almost unique
feature_candidates = [c for c in feature_candidates if df[c].nunique() < 0.95*len(df)]
# Drop coordinates or free-text if named suspiciously
feature_candidates = [c for c in feature_candidates if not any(c.lower().startswith(p) for p in exclude_prefixes)]
print(f"\nSelected {len(feature_candidates)} candidate features (preview):")
print(feature_candidates[:40])

# For demonstration we will use:
# - All categorical (object / low-cardinality) columns
# - numeric columns
# - limit one-hot to columns with <= 30 unique categories to avoid explosion
X = df[feature_candidates].copy()

# Remove columns that are identical to the target or contain target-like leakage
# e.g., columns with 'police' in name except maybe 'location' etc.
leak_cols = [c for c in X.columns if 'police' in c.lower() and c != TARGET_COL]
if leak_cols:
    print("\nDropping columns that leak the target or contain 'police' in their name:", leak_cols)
    X = X.drop(columns=leak_cols)

# Identify numeric vs categorical
numeric_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object','category','bool']).columns.tolist()

# Filter categorical to manageable cardinality
cat_keep = []
for c in categorical_cols:
    nun = X[c].nunique(dropna=True)
    if nun <= 30:
        cat_keep.append(c)
    else:
        # try to reduce by extracting coarse values (e.g., take first token)
        X[c] = X[c].fillna("missing").astype(str).str[:80]
        if X[c].nunique() <= 60:
            cat_keep.append(c)
# final categorical
categorical_cols = cat_keep

print(f"\nUsing numeric cols ({len(numeric_cols)}): {numeric_cols[:10]}")
print(f"Using categorical cols ({len(categorical_cols)}): {categorical_cols[:20]}")

# If no feature columns found, exit
if len(numeric_cols) + len(categorical_cols) == 0:
    print("No usable features found automatically. Please select features manually.")
    sys.exit(1)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X[numeric_cols + categorical_cols], y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"\nTrain shape: {X_train.shape}, Test shape: {X_test.shape}")

# Build preprocessing pipelines
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols)
], remainder='drop')

# Helper: fit model, evaluate
def fit_evaluate_model(clf, name, X_train, X_test, y_train, y_test, preprocessor):
    print(f"\n=== Training {name} ===")
    pipe = Pipeline([
        ('preproc', preprocessor),
        ('clf', clf)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:,1] if hasattr(pipe.named_steps['clf'], "predict_proba") else None

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_proba) if y_proba is not None else float('nan')

    print(f"{name} -- Accuracy: {acc:.4f}, F1: {f1:.4f}, ROC-AUC: {roc:.4f}")
    print("Classification report:")
    print(classification_report(y_test, y_pred))
    return pipe, dict(accuracy=acc, f1=f1, roc_auc=roc, y_pred=y_pred, y_proba=y_proba)

# 1) Logistic Regression (with L2)
lr = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, class_weight='balanced')
lr_pipe, lr_metrics = fit_evaluate_model(lr, "LogisticRegression", X_train, X_test, y_train, y_test, preprocessor)

# 2) Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, class_weight='balanced', n_jobs=-1)
rf_pipe, rf_metrics = fit_evaluate_model(rf, "RandomForest", X_train, X_test, y_train, y_test, preprocessor)

# 3) XGBoost (if available)
use_xgb = True
try:
    import xgboost as xgb
    xgb_clf = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=RANDOM_STATE, n_estimators=200)
    xgb_pipe, xgb_metrics = fit_evaluate_model(xgb_clf, "XGBoost", X_train, X_test, y_train, y_test, preprocessor)
except Exception as e:
    print("\nXGBoost not available or failed to import:", e)
    use_xgb = False
    xgb_pipe, xgb_metrics = None, None

# Compare models
results = {
    'LogisticRegression': lr_metrics,
    'RandomForest': rf_metrics
}
if use_xgb:
    results['XGBoost'] = xgb_metrics

print("\n=== Summary of results ===")
for k,v in results.items():
    print(f"{k}: Accuracy={v['accuracy']:.4f}, F1={v['f1']:.4f}, ROC-AUC={v['roc_auc']:.4f}")

# Choose best model by F1 (or ROC if you prefer)
best_model_name = max(results.items(), key=lambda kv: (kv[1]['f1'], kv[1]['roc_auc']))[0]
print(f"\nSelected best model: {best_model_name}")

best_pipe = {'LogisticRegression': lr_pipe, 'RandomForest': rf_pipe, 'XGBoost': xgb_pipe}[best_model_name]

# --------------------- SHAP explainability ---------------------
# Compute SHAP feature importance for tree-based models or approximate for LR
try:
    import shap
    print("\nComputing SHAP values for the best model...")
    # We need the preprocessor and the classifier separately to compute shap on transformed features
    # We'll transform a sample of training data
    X_sample = X_train.sample(min(2000, len(X_train)), random_state=RANDOM_STATE)
    X_trans = preprocessor.transform(X_sample)

    # Get feature names for transformed matrix
    # numeric -> same names; categorical -> onehot names
    num_names = numeric_cols
    cat_names = []
    if categorical_cols:
        ohe = preprocessor.named_transformers_['cat'].named_steps['onehot']
        ohe_cols = ohe.get_feature_names_out(categorical_cols)
        cat_names = list(ohe_cols)
    feature_names = list(num_names) + cat_names

    # SHAP for XGBoost or RandomForest via TreeExplainer; for LR use KernelExplainer (slower)
    if best_model_name == 'XGBoost' and use_xgb:
        # access the underlying xgb model
        model_for_shap = best_pipe.named_steps['clf']
        explainer = shap.TreeExplainer(model_for_shap)
        shap_values = explainer.shap_values(X_trans)
        # plot summary
        shap.summary_plot(shap_values, X_trans, feature_names=feature_names, show=True)
    elif best_model_name == 'RandomForest':
        model_for_shap = best_pipe.named_steps['clf']
        explainer = shap.TreeExplainer(model_for_shap)
        shap_values = explainer.shap_values(X_trans)
        # If classifier shap_values is list (for each class), take class 1
        if isinstance(shap_values, list):
            shap_vals = shap_values[1]
        else:
            shap_vals = shap_values
        shap.summary_plot(shap_vals, X_trans, feature_names=feature_names, show=True)
    else:
        # Logistic Regression: use Linear shap (approx using shap.LinearExplainer if supported)
        model_for_shap = best_pipe.named_steps['clf']
        try:
            explainer = shap.LinearExplainer(model_for_shap, X_trans, feature_perturbation="interventional")
            shap_values = explainer.shap_values(X_trans)
            shap.summary_plot(shap_values, X_trans, feature_names=feature_names, show=True)
        except Exception as e:
            print("SHAP for LogisticRegression failed or is slow. Skipping SHAP:", e)
except Exception as e:
    print("\nSHAP not available or failed. Install shap to get feature importance plots. Error:", e)

# --------------------- Error analysis / subgroup performance ---------------------
print("\nRunning subgroup error analysis...")

# Candidate subgroup columns to analyze (common contextual/demographic columns)
subgroup_candidates = [c for c in df.columns if any(k in c.lower() for k in ['location','place','intent','race','age','sex','gender','setting','city','state','county','neighborhood'])]
# Keep only ones we actually used as features (so they exist in X)
subgroup_cols = [c for c in subgroup_candidates if c in X.columns]

print("Subgroup columns found for analysis:", subgroup_cols)

# Use predictions from best model
y_pred = best_pipe.predict(X_test)
y_proba = best_pipe.predict_proba(X_test)[:,1] if hasattr(best_pipe.named_steps['clf'], "predict_proba") else None

global_cm = confusion_matrix(y_test, y_pred)
print("\nGlobal confusion matrix (rows true, cols pred):")
print(global_cm)

# Compute false negative rate (FNR) = FN / (FN + TP) for each subgroup value
def subgroup_fnr_analysis(X_test_df, y_true, y_pred, subgroup_col):
    df_local = X_test_df.copy()
    df_local['_y_true'] = y_true.values
    df_local['_y_pred'] = y_pred
    groups = df_local.groupby(subgroup_col)
    rows = []
    for name, g in groups:
        tn, fp, fn, tp = confusion_matrix(g['_y_true'], g['_y_pred'], labels=[0,1]).ravel()
        support = len(g)
        # Avoid division by zero
        fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
        tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        rows.append({'subgroup_col':subgroup_col, 'subgroup_value':name, 'support':support, 'TP':tp, 'FN':fn, 'FNR':fnr, 'TPR':tpr})
    out = pd.DataFrame(rows).sort_values(['support'], ascending=False)
    return out

# For each subgroup column, compute FNR table and report top underpredicted values (highest FNR, support >= 30)
for col in subgroup_cols:
    print(f"\nAnalyzing subgroup: {col}")
    tbl = subgroup_fnr_analysis(X_test.reset_index(drop=True), y_test.reset_index(drop=True), y_pred, col)
    # filter for reasonable support
    tbl_filt = tbl[tbl['support'] >= max(10, 0.01 * len(X_test))].copy()
    if tbl_filt.empty:
        print(" - No subgroup value with enough support to analyze.")
        continue
    # Rank by FNR descending
    tbl_filt = tbl_filt.sort_values('FNR', ascending=False)
    print("Top underpredicted subgroup values (high FNR):")
    print(tbl_filt[['subgroup_value', 'support', 'TP', 'FN', 'FNR']].head(10).to_string(index=False))

    # Plot FNR vs support (scatter)
    plt.figure(figsize=(8,5))
    sns.scatterplot(x='support', y='FNR', data=tbl_filt)
    plt.title(f"Subgroup FNR by support for {col}")
    plt.xlabel("Support (n)")
    plt.ylabel("False Negative Rate")
    plt.ylim(0,1)
    plt.grid(True)
    plt.show()

# If no subgroup columns found, do simpler analysis by 'location_type' if available
if not subgroup_cols:
    print("\nNo obvious subgroup columns detected. As a fallback, try 'location' or 'place' values in original data.")
    for alt in ['location','place','location_type','setting']:
        if alt in df.columns:
            print(f"Using {alt} for subgroup analysis.")
            tbl = subgroup_fnr_analysis(df.loc[X_test.index, [alt]], y_test.reset_index(drop=True), y_pred, alt)
            print(tbl.sort_values('FNR', ascending=False).head(10))
            break

# --------------------- Save models / artifacts (optional) ---------------------
# You can save the best pipeline with joblib if desired:
try:
    import joblib
    joblib.dump(best_pipe, f"best_model_{best_model_name}.joblib")
    print(f"\nSaved best model pipeline to best_model_{best_model_name}.joblib")
except Exception as e:
    print("joblib not available or save failed:", e)

print("\nDone. Review printed metrics and plots. For further help (hyperparameter tuning, calibration, fairness interventions), ask me and I can extend this pipeline.")
